# Figure S7: Stage1 Latent Clustering

- Figure / panels: `FigS7a`, `FigS7b`, `FigS7c`, `FigS7d`
- Inputs: `artifacts/stage1_latent_clustering/pbmc_celltype/pbmc/split1/train_all_cells_seed24/*`
- Fallback: if the PBMC artifact is missing, rerun `run_stage1_latent_clustering(...)` with the same config
- Outputs: `artifacts/paper_figures/supp/FigS7_Stage1LatentClustering/*`
- Run order: supplementary-only; does not affect the main benchmark notebooks
- Role: support the claim that Stage1 learns biologically meaningful cell representations


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis
from scripts.trishift.analysis.stage1_latent_clustering import run_stage1_latent_clustering

apply_gears_paper_style(font_scale=1.0)


In [ ]:
MODE = 'pbmc_celltype'  # fixed for the paper supplement
DATASET_NAME = 'pbmc'
SPLIT_ID = 1
STAGE1_POOL_MODE = 'train_all_cells'
RANDOM_SEED = 24

SOURCE_ROOT = repo_root / 'artifacts' / 'stage1_latent_clustering' / MODE / DATASET_NAME / f'split{SPLIT_ID}' / f'{STAGE1_POOL_MODE}_seed{RANDOM_SEED}'
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS7_Stage1LatentClustering'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Preferred source:', SOURCE_ROOT)
print('Output root:', OUT_ROOT)


In [ ]:
if SOURCE_ROOT.exists() and (SOURCE_ROOT / 'cluster_metrics.csv').exists():
    source_dir = SOURCE_ROOT
else:
    result = run_stage1_latent_clustering(
        mode=MODE,
        dataset_name=DATASET_NAME,
        split_id=SPLIT_ID,
        stage1_pool_mode=STAGE1_POOL_MODE,
        random_seed=RANDOM_SEED,
    )
    source_dir = Path(result.out_dir)

metrics_df = pd.read_csv(source_dir / 'cluster_metrics.csv')
run_meta = json.loads((source_dir / 'run_meta.json').read_text(encoding='utf-8'))
metrics_df.to_csv(OUT_ROOT / 'figs7_cluster_metrics.csv', index=False, encoding='utf-8-sig')
(OUT_ROOT / 'figs7_source_run_meta.json').write_text(json.dumps(run_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print('Resolved source:', source_dir)
display(metrics_df)


In [ ]:
figure_sources = {
    'figs7a_umap_by_cluster.png': source_dir / 'umap_by_cluster.png',
    'figs7b_umap_by_cell_type.png': source_dir / 'umap_by_label_cell_type.png',
    'figs7c_cluster_vs_cell_type.png': source_dir / 'cluster_vs_label_cell_type.png',
}
for out_name, src_path in figure_sources.items():
    if src_path.exists():
        shutil.copy2(src_path, OUT_ROOT / out_name)

metric_row = metrics_df[metrics_df['label_key'].astype(str) == 'label_cell_type'].copy()
metric_cols = [col for col in ['ARI_cluster/label', 'NMI_cluster/label', 'ASW_label', 'avg_bio'] if col in metric_row.columns]
plot_df = metric_row[metric_cols].melt(var_name='metric', value_name='value') if not metric_row.empty and metric_cols else pd.DataFrame(columns=['metric', 'value'])
fig, ax = plt.subplots(figsize=(7.0, 4.2), dpi=220)
if plot_df.empty:
    ax.text(0.5, 0.5, 'No PBMC cell-type metric summary available', ha='center', va='center')
    ax.axis('off')
else:
    sns.barplot(data=plot_df, x='metric', y='value', color='#4C78A8', ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_title('FigS7d: Stage1 PBMC clustering metrics')
    ax.tick_params(axis='x', rotation=25)
    style_axis(ax, grid_axis='y')
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_linewidth(0.5)
fig.tight_layout()
fig.savefig(OUT_ROOT / 'figs7d_cluster_metrics.png', bbox_inches='tight')
plt.close(fig)
plot_df.to_csv(OUT_ROOT / 'figs7d_cluster_metrics_values.csv', index=False, encoding='utf-8-sig')


In [ ]:
for image_name in [
    'figs7a_umap_by_cluster.png',
    'figs7b_umap_by_cell_type.png',
    'figs7c_cluster_vs_cell_type.png',
    'figs7d_cluster_metrics.png',
]:
    image_path = OUT_ROOT / image_name
    print(image_path)
    if image_path.exists():
        display(Image(filename=str(image_path), width=900))
print(OUT_ROOT)
